# Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import pydicom
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

In [ ]:
#seed used for random
seed = 42
torch.manual_seed(seed)

# Loading measurement data

In [ ]:
#read csv
df = pd.read_csv("../data/measurements.csv")

print(df.columns)

#drop unnecessary columns
df.drop(columns = ["Student Name", "Notes", "Unnamed: 23", "Unnamed: 24", "Unnamed: 25", "Unnamed: 26"], inplace = True)

print(df.columns)

print(len(df.columns.values))

In [ ]:
#get rid of images with na (only temporary for convenience)
df = df.dropna()

# Load image data (datasets and dataloaders)

In [ ]:
#image directory
img_dir = "../data/box_images"

#train test split
train_test_split = 0.8

#batch size
batch_size = 8

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, img_dir, img_files, img_transform = None):
        self.img_dir = img_dir
        self.img_files = img_files
        self.img_transform = img_transform

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_dir + '/' + self.img_files[idx]
        ds = pydicom.dcmread(img_path)
        img = ds.pixel_array

        img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
        img = img.astype(np.uint8)

        img = Image.fromarray(img)

        if self.img_transform:
            img = self.img_transform(img)
        
        return img, self.img_files[idx]
        

In [ ]:
#all images in directory
all_img_in_dir = [i for i in os.listdir(img_dir) if i.lower().endswith("dcm")]

#check all images in df are loaded (else throw error)
for i in df['ID'].values:
    if i not in all_img_in_dir:
        raise SystemError(f'Directory is missing images: {i}')
    
#load only images in the df
all_img = [i for i in all_img_in_dir if i in df['ID'].values]
print(f'Images in use: {len(all_img)}/{len(all_img_in_dir)}')

#split train and test sets
all_idx = torch.randperm(len(all_img))
train_img = [all_img[i] for i in all_idx[:int(train_test_split * len(all_img))]]
test_img = [all_img[i] for i in all_idx[int(train_test_split * len(all_img)):]]

In [ ]:
#transform used for training
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

#transform used for testing
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

In [ ]:
#training dataset and data loader
train_set = ImageDataset(img_dir, train_img, train_transform)
train_loader = DataLoader(train_set, batch_size = batch_size, shuffle = True)

#testing dataset and data loader
test_set = ImageDataset(img_dir, test_img, test_transform)
test_loader = DataLoader(test_set, batch_size = batch_size, shuffle = False)

In [ ]:
for imgs, files in train_loader:
    print(files)
    plt.imshow(imgs[0].squeeze(), cmap = 'gray')
    plt.axis('off')
    plt.show()


# Model

In [ ]:
#number of epochs
epoch_cnt = 20

#learning rate
learning_rate = 1e-3

In [ ]:
#model
class CNNModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size = 3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 20)
        )

    def forward(self, x):
        x = self.seq(x)
        return x

model = CNNModel()

In [ ]:
#device
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

In [ ]:
#loss function
lossfn = nn.MSELoss()

In [ ]:
#optimizer
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

In [ ]:
for epoch in range(epoch_cnt):
    model.to(device)
    model.train()

    total_loss = 0
    
    for images, files in train_loader:
        images = images.to(device)
        
        yvals = []
        for file in files:
            yvals.append(df.loc[df['ID'] == file].drop(columns = 'ID').values.astype(np.float32))
        yvals = torch.from_numpy(np.stack(yvals)).squeeze(1).to(device)

        ypred = model(images)

        loss = lossfn(ypred, yvals)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    
    # if epoch % 10 and epoch != epoch_cnt - 1: #only print every x epochs
    #     continue

    print(f'Epoch: {epoch}\n')
    print(f'Loss (in sample): {total_loss / len(train_set)}')

    total_loss = 0
    total_percent_err = torch.zeros(20).to(device)
    model.eval()
    for images, files in test_loader:
        images = images.to(device)
        
        yvals = []
        for file in files:
            yvals.append(df.loc[df['ID'] == file].drop(columns = 'ID').values.astype(np.float32))
        yvals = torch.from_numpy(np.stack(yvals)).squeeze(1).to(device)

        ypred = model(images)

        loss = lossfn(ypred, yvals)
        total_loss += loss.item()

        total_percent_err += torch.sum(torch.abs(yvals - ypred) / yvals, dim = 0)

        if epoch == epoch_cnt - 1:
            plt.imshow(images[0].squeeze().cpu(), cmap = 'gray')
            plt.axis('off')
            plt.show()
            print(ypred[0])
            print(yvals[0])
    
    print(f'Loss (out sample): {total_loss / len(test_set)}\n')
    print(f'Percent error for each measurement (out sample):')
    for i in total_percent_err:
        print(f'{(i.item() / len(test_set)):.4f}', end = ' ')

    print('\n===\n')




            